<a href="https://colab.research.google.com/github/Panperception/UG_2025_QRC/blob/main/lnn_tensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tensorized Liquid Neural Network (TLNN)

This notebook implements a Liquid Neural Network where the weight matrices
are replaced by Tensor-Train decomposed layers.

Advantages:
- drastically fewer parameters
- better scalability
- structural similarity to tensor networks

In [ ]:
!pip install torch torchvision torchdiffeq

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from torchdiffeq import odeint

## Use GPU if available

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Load MNIST dataset

Each 28×28 image is treated as a sequence of 28 time steps.

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

## Tensor-Train Layer

Instead of a dense matrix W ∈ R^(n×n), we represent it as a
product of smaller tensor cores.

In [ ]:
class TTLinear(nn.Module):

    def __init__(self, in_features, out_features, rank=8):
        super().__init__()

        self.core1 = nn.Parameter(
            torch.randn(in_features, rank)
        )

        self.core2 = nn.Parameter(
            torch.randn(rank, out_features)
        )

        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):

        x = torch.matmul(x, self.core1)

        x = torch.matmul(x, self.core2)

        return x + self.bias

## Tensorized Liquid Dynamics

In [ ]:
class TensorLiquidODEFunc(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.W = TTLinear(hidden_size, hidden_size)

        self.U = TTLinear(input_size, hidden_size)

        self.W_tau = TTLinear(hidden_size, hidden_size)

        self.U_tau = TTLinear(input_size, hidden_size)

        self.activation = torch.tanh

        self.current_input = None

    def set_input(self, u):
        self.current_input = u

    def forward(self, t, x):

        u = self.current_input

        tau = torch.sigmoid(
            self.W_tau(x) + self.U_tau(u)
        ) + 0.1

        dxdt = -x / tau + self.activation(
            self.W(x) + self.U(u)
        )

        return dxdt

## Tensorized Liquid Layer

In [ ]:
class TensorLiquidLayer(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.func = TensorLiquidODEFunc(
            input_size,
            hidden_size
        )

        self.hidden_size = hidden_size

    def forward(self, x_seq):

        batch_size, seq_len, _ = x_seq.shape

        h = torch.zeros(batch_size, self.hidden_size).to(x_seq.device)

        t = torch.tensor([0,1]).float().to(x_seq.device)

        for step in range(seq_len):

            u = x_seq[:,step,:]

            self.func.set_input(u)

            h = odeint(self.func, h, t)[-1]

        return h

# Tensorized Model

In [ ]:
class TensorLiquidModel(nn.Module):

    def __init__(self, input_size=28, hidden_size=64, num_classes=10):

        super().__init__()

        self.liquid = TensorLiquidLayer(
            input_size,
            hidden_size
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        x = x.squeeze(1)

        h = self.liquid(x)

        out = self.fc(h)

        return out


## Training Setup

In [ ]:
model = TensorLiquidModel().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Training

In [ ]:
epochs = 5

for epoch in range(epochs):

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:",epoch+1,"Loss:",total_loss/len(train_loader))

## Evaluation

In [ ]:
test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transform
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Test Accuracy:",100*correct/total)